In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
from src.morse_dataset import MorseDataset
from models.crnn import MorseCRNN1D
from src.train_model import train_model
from src.collate_fn import collate_fn, test_collate_fn
from src.encoding_decoding import decode_predictions, build_char_idx_dict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
df_train_full = pd.read_csv("data/train/labels.csv")
df_train_full["filename"] = df_train_full["filename"].map(lambda x: x.replace(".wav", ".pt"))
df_train_full

,filename,text
0,00000.pt,83
1,00001.pt,68
2,00002.pt,41254
3,00003.pt,27-9
4,00004.pt,55-3991
...,...,...
29995,29995.pt,915
29996,29996.pt,78922
29997,29997.pt,611
29998,29998.pt,5792


In [3]:
alphabet = "0123456789- "
char_to_idx, idx_to_char = build_char_idx_dict(alphabet=alphabet)

train_files = ["data/train_specs_2/" + str(f) for f in df_train_full["filename"].values]
train_labels = df_train_full["text"].astype(str).values.tolist()

train_dataset = MorseDataset(train_files, train_labels, char_to_idx)

In [4]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=collate_fn
)

In [5]:
num_classes = len(char_to_idx)
model = MorseCRNN1D(num_classes=num_classes).to(device)

best_model_path = "models_files/morse_crnn_40_best.pt"
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

model, history = train_model(
    model,
    criterion,
    optimizer,
    train_loader,
    epochs=20,
    device=device,
    scheduler=scheduler,
    save_path="morse_crnn_full_train",
)

Epoch 1/20 | train_loss: 0.1730
Epoch 2/20 | train_loss: 0.2486
Epoch 3/20 | train_loss: 0.1659
Epoch 4/20 | train_loss: 0.1379
Epoch 5/20 | train_loss: 0.1254
Epoch 6/20 | train_loss: 0.1204
Epoch 7/20 | train_loss: 0.1137
Epoch 8/20 | train_loss: 0.1396
Epoch 9/20 | train_loss: 0.1435
Epoch 10/20 | train_loss: 0.1585
Epoch 11/20 | train_loss: 0.1073
Epoch 12/20 | train_loss: 0.0884
Epoch 13/20 | train_loss: 0.0827
Epoch 14/20 | train_loss: 0.0831
Epoch 15/20 | train_loss: 0.0831
Epoch 16/20 | train_loss: 0.0938
Epoch 17/20 | train_loss: 0.0795
Epoch 18/20 | train_loss: 0.0718
Epoch 19/20 | train_loss: 0.0635
Epoch 20/20 | train_loss: 0.0579


In [9]:
submission_df = pd.read_csv("sample_submission.csv")
submission_df["filename"] = submission_df["filename"].map(lambda x: x.replace(".wav", ".pt"))

test_files = ["data/test_specs_2/" + str(f) for f in submission_df["filename"].values]
test_labels = [""] * len(test_files)

test_dataset = MorseDataset(test_files, test_labels, char_to_idx)
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=test_collate_fn,
)

In [10]:
num_classes = len(char_to_idx)
model = MorseCRNN1D(num_classes=num_classes).to(device)

best_model_path = "morse_crnn_full_train_20.pt"
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

pred_texts = []

with torch.no_grad():
    for inputs, input_lengths in test_loader:
        inputs = inputs.to(device)

        outputs = model(inputs)

        if outputs.size(0) == inputs.size(0):
            outputs = outputs.transpose(0, 1)

        pred_ids = outputs.argmax(dim=2).cpu()

        output_time = outputs.size(0)
        input_time = inputs.size(-1)

        for i, length in enumerate(input_lengths):
            ids = pred_ids[:length, i].tolist()
            text = decode_predictions(ids, idx_to_char)
            pred_texts.append(text)

submission_df["text"] = pred_texts
submission_df["filename"] = submission_df["filename"].map(lambda x: x.replace('.pt', '.wav'))
submission_df.to_csv("submission_crnn_full_train.csv", index=False)